# Transformer kernel — pass 3 verification + follow-ups (Tesla T4)

The pass-3 experiment matrix E1–E6 already ran on the pre-merge branch (raw data committed as `results/pass3-e*.json`; findings in the README's pass-3 section and `docs/pass3-research.md`). This notebook targets the **merged** tree and answers what E1–E6 left open:

* **F1** — kernel tests + best-config regression sweep on the merged code
* **F2** — case 7: plain defaults vs `torch.compile reduce-overhead`, head to head in one session
* **F3** — case 6 seed-robustness: what `--fp16-max-elements` (fp32 dispatch for oversized forwards) costs, and whether it survives the 25-trial × 2-seed stress
* **F4** — the experimental Triton attention kernel's negative result, reproduced post-merge

Repo: https://github.com/danielfodgaard/transformer-kernel — every run writes JSON into `results/`; the last cells print a summary and pack `results.tar.gz`.

In [ ]:
BRANCH = "claude/transformer-kernel-gpu-optimization-kq9ns1"  # drop the checkout once pass 3 is merged to main

!nvidia-smi --query-gpu=name,temperature.gpu,clocks.sm --format=csv
!cd /kaggle/working && rm -rf transformer-kernel && \
  git clone -q https://github.com/danielfodgaard/transformer-kernel.git && \
  cd transformer-kernel && git checkout -q {BRANCH} && git log --oneline -1
import torch
print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

## F1 — kernel tests, then the best-config regression sweep

Fail-early gate: the GPU test script covers the shipping fused-LN kernels; the pytest suite additionally covers the experimental attention kernel (head_dim 8→256, ragged S, strided QKV views). Then the measured-best per-case configuration (`configs/best.json`: CUDA graphs on 1–4 and 12, plain fused defaults elsewhere) reruns on the merged tree — every case should land near the pass-2 table (geomean 7.11x) — and case 14 gets its out-of-core quick pass.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/test_kernels.py
!cd /kaggle/working/transformer-kernel && python -m pytest src/test_triton_kernels.py -q

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --config configs/best.json --skip 14 \
  --out results/pass3f-best-regression.json
!cd /kaggle/working/transformer-kernel && python src/run_case14.py --max-samples 4

## F2 — case 7 head-to-head: defaults vs compile

Cross-session hint from E5: compiled `reduce-overhead` measured 1.46 ms where the pass-2 session's plain defaults measured 1.60 ms. Same session, same clocks, winner takes the `configs/best.json` entry.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 7 \
  --out results/pass3f-case7-default.json
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 7 \
  --out results/pass3f-case7-compiled.json -- --compile-user --compile-mode reduce-overhead

## F3 — case 6 seed-robustness option

Case 6 in fp16 fails a 25-trial stress on seed 1234 (0.002074) and grazes 0.0022 on seed 9999 — reproduced in two sessions. `--fp16-max-elements 100000000` dispatches forwards bigger than 100M elements (case 6 is 164M) to fp32. First cell prices that option; the stress cells check it actually buys seed-robustness (expect ~1e-6 error class). The default stays fp16 unless the price is acceptable — this is the data that decision needs.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 \
  --out results/pass3f-case6-fp32dispatch.json -- --fp16-max-elements 100000000
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 \
  --out results/pass3f-case6-fp32dispatch-stress-a.json -- --fp16-max-elements 100000000 \
  --accuracy-trials 25 --warmup 1 --repeats 1 --benchmark-rounds 1
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 6 \
  --out results/pass3f-case6-fp32dispatch-stress-b.json -- --fp16-max-elements 100000000 \
  --accuracy-trials 25 --seed 9999 --warmup 1 --repeats 1 --benchmark-rounds 1

## F4 — attention-kernel negative result, post-merge

Cheap reproduction that `--attention triton` still dispatches correctly on the merged tree and still loses to SDPA (E2 measured case 11 at 7.47 ms vs 3.47 ms). If it ever starts *winning* after a Triton/torch upgrade, this cell is how we notice.

In [ ]:
!cd /kaggle/working/transformer-kernel && python src/sweep.py --cases 11,13 \
  --out results/pass3f-attn-triton.json -- --attention triton

In [ ]:
# Summary: follow-up runs vs the pass-2 measured-best table.
import json, pathlib

PASS2_BEST = {1: 7.053, 2: 9.545, 3: 9.186, 4: 6.636, 5: 6.981, 6: 7.348,
              7: 3.932, 8: 5.087, 9: 4.434, 10: 5.244, 11: 10.193,
              12: 6.949, 13: 16.977}

root = pathlib.Path('/kaggle/working/transformer-kernel/results')
for path in sorted(root.glob('pass3f-*.json')):
    data = json.loads(path.read_text())
    print(f"\n=== {path.stem.replace('pass3f-', '')} | args={data['passthrough_args']}")
    for case in data['cases']:
        cid = case['case']['id']
        acc = case.get('accuracy') or {}
        speed = f"{case['speedup']:.3f}x" if case.get('speedup') else '-'
        max_abs = acc.get('max_abs_error')
        err = f"{max_abs:.2e}" if max_abs is not None else '-'
        ref = f"  (pass-2 best {PASS2_BEST[cid]:.2f}x)" if cid in PASS2_BEST else ''
        print(f"  case {cid:>2} {case['status']:<16} {speed:>9}  max_abs={err}{ref}")

In [ ]:
!cd /kaggle/working/transformer-kernel && tar czf /kaggle/working/results.tar.gz results/
print("Download results.tar.gz from the notebook's Output panel")